In [ ]:
newsapi='a752bdf354ee4625b3ed58d906eec969'

import requests
def get_news():
    url = f'https://newsapi.org/v2/top-headlines?country=us&apiKey={newsapi}'
    params = {
            "q" : "conflict",
            "sortBy" : "publishedAt",
            "language" : "en",
            "pageSize" : 10,
            "apiKey" : newsapi,
            }
    response = requests.get(url, params=params)
    data = response.json()
    return data['articles']

response = get_news()
for article in response:
    print(f"Title: {article['title']}")
    print(f"Description: {article['description']}")
    print(f"URL: {article['url']}")
    print(f"Published At: {article['publishedAt']}")
    print("-" * 80)
import pandas 
df = pandas.DataFrame(response)

Title: Lindsay Lohan, Family 'Safe' Amid Escalating Conflict in Middle East - TMZ
Description: Lindsay Lohan and her family are okay following the strikes on Dubai by Iran ... TMZ has learned.
URL: https://www.tmz.com/2026/03/01/lindsay-lohan-family-safe-after-dubai-bombings/
Published At: 2026-03-02T04:22:56Z
--------------------------------------------------------------------------------


,source,author,title,description,url,urlToImage,publishedAt,content
0,"{'id': None, 'name': 'TMZ'}",TMZ Staff,"Lindsay Lohan, Family 'Safe' Amid Escalating C...",Lindsay Lohan and her family are okay followin...,https://www.tmz.com/2026/03/01/lindsay-lohan-f...,https://imagez.tmz.com/image/58/16by9/2026/03/...,2026-03-02T04:22:56Z,Lindsay Lohan and her family are okay followin...


In [14]:
len(df['description'][0])

97

In [ ]:
import feedparser
import pandas as pd
from datetime import datetime

# -----------------------------
# 1️⃣ News Sources (RSS)
# -----------------------------
RSS_SOURCES = {
    "BBC": "https://feeds.bbci.co.uk/news/world/rss.xml",
    "Reuters": "https://www.reuters.com/world/rss.xml",
    "Guardian": "https://www.theguardian.com/world/rss",
    "AlJazeera": "https://www.aljazeera.com/xml/rss/all.xml",
    "CNN": "http://rss.cnn.com/rss/edition_world.rss"
}

# -----------------------------
# 2️⃣ Conflict Keywords
# -----------------------------
KEYWORDS = [
    "war",
    "world war",
    "iran",
    "israel",
    "united states",
    "us",
    "missile",
    "attack",
    "conflict",
    "retaliation",
    "military",
    "airstrike",
    "tehran",
    "gaza",
    "hezbollah"
]

# -----------------------------
# 3️⃣ Function to Check Relevance
# -----------------------------
def is_conflict_related(text):
    text = text.lower()
    return any(keyword in text for keyword in KEYWORDS)

# -----------------------------
# 4️⃣ Scraping Function
# -----------------------------
def scrape_conflict_news():
    all_articles = []

    for source_name, url in RSS_SOURCES.items():
        print(f"Scraping {source_name}...")
        feed = feedparser.parse(url)

        for entry in feed.entries:
            title = entry.title
            summary = entry.get("summary", "")
            link = entry.link
            published = entry.get("published", "")

            combined_text = f"{title} {summary}"

            if is_conflict_related(combined_text):
                all_articles.append({
                    "source": source_name,
                    "title": title,
                    "summary": summary,
                    "published": published,
                    "link": link,
                    "scraped_at": datetime.now()
                })

    return pd.DataFrame(all_articles)

# -----------------------------
# 5️⃣ Run Scraper
# -----------------------------
if __name__ == "__main__":
    df = scrape_conflict_news()

    print("\nTotal Conflict Articles Found:", len(df))
    print(df.head())

    # Save to CSV
    df.to_csv("conflict_news.csv", index=False)
    

Scraping BBC...
Scraping Reuters...
Scraping Guardian...
Scraping AlJazeera...
Scraping CNN...

Total Conflict Articles Found: 98
  source                                              title  \
0    BBC  Allies of US in the Gulf bear brunt of Iran at...   
1    BBC  Oil prices rise after ships attacked near Stra...   
2    BBC  Michael B Jordan upends Oscars race as Sinners...   
3    BBC  At least 153 dead after reported strike on sch...   
4    BBC  Nine dead in missile attack on Israel as Iran ...   

                                             summary  \
0  Iran's attacks on Gulf Arab states suggest the...   
1  Experts have warned that a prolonged conflict ...   
2  The US star's best actor win for Sinners leave...   
3  Iran has blamed the US and Israel for the stri...   
4  Several deaths are reported across the Middle ...   

                       published  \
0  Sun, 01 Mar 2026 22:19:03 GMT   
1  Mon, 02 Mar 2026 06:53:43 GMT   
2  Mon, 02 Mar 2026 05:08:35 GMT   
3  Sun, 01

In [3]:
df.head()

,source,title,summary,published,link,scraped_at
0,BBC,Allies of US in the Gulf bear brunt of Iran at...,Iran's attacks on Gulf Arab states suggest the...,"Sun, 01 Mar 2026 22:19:03 GMT",https://www.bbc.com/news/articles/c1jk922dgjgo...,2026-03-02 12:53:54.525642
1,BBC,Oil prices rise after ships attacked near Stra...,Experts have warned that a prolonged conflict ...,"Mon, 02 Mar 2026 06:53:43 GMT",https://www.bbc.com/news/articles/c75evve6l63o...,2026-03-02 12:53:54.525655
2,BBC,Michael B Jordan upends Oscars race as Sinners...,The US star's best actor win for Sinners leave...,"Mon, 02 Mar 2026 05:08:35 GMT",https://www.bbc.com/news/articles/clyzklvk79yo...,2026-03-02 12:53:54.525664
3,BBC,At least 153 dead after reported strike on sch...,Iran has blamed the US and Israel for the stri...,"Sun, 01 Mar 2026 15:32:26 GMT",https://www.bbc.com/news/articles/c1l7rvqq51eo...,2026-03-02 12:53:54.525666
4,BBC,Nine dead in missile attack on Israel as Iran ...,Several deaths are reported across the Middle ...,"Sun, 01 Mar 2026 22:24:27 GMT",https://www.bbc.com/news/articles/c363zkp1pgxo...,2026-03-02 12:53:54.525668


In [2]:
from ast import Global

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

RSS_SOURCES = {
    "BBC": "https://feeds.bbci.co.uk/news/rss.xml",
    "Associated Press": "https://feedx.net/rss/ap.xml",
    "UN News": "https://news.un.org/en/rss-feeds",
    "Al Jazeera": "https://www.aljazeera.com/xml/rss/all.xml",
    "Reuters": "https://www.reutersagency.com/feed/?best-topics=top-news",
    "Guardian": "https://www.theguardian.com/world/rss",
    "CNN": "http://rss.cnn.com/rss/edition_world.rss",
    "POLITICO": "https://www.politico.com/rss",
    "NYTimes": "https://www.nytimes.com/rss",
    "NDTV": "https://www.ndtv.com/rss",
    "The Hindu": "https://www.thehindu.com/rssfeeds/",
    "Newswise": "https://www.newswise.com/channels/rss"
}

KEYWORDS = [
    "war", "iran", "israel", "united states",
    "us", "missile", "attack", "military",
    "retaliation", "conflict", "gaza",
    "tehran", "hezbollah"
]


def is_conflict_related(text):
    text = text.lower()
    return any(k in text for k in KEYWORDS)


def extract_full_article(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = soup.find_all("p")
        article_text = " ".join([p.get_text() for p in paragraphs])
        #print(paragraphs)
        return article_text
    
    except Exception as e:
        print("Error scraping:", url)
        return ""


def scrape_conflict_news():
    all_articles = []

    for source_name, rss_url in RSS_SOURCES.items():
        print(f"\nScraping {source_name}...")

        feed = feedparser.parse(rss_url)

        for entry in feed.entries:
            title = entry.title
            link = entry.link
            published = entry.get("published", "")

            full_text = extract_full_article(link)

            if is_conflict_related(title + full_text):
                all_articles.append({
                    "source": source_name,
                    "title": title,
                    "published": published,
                    "content": full_text,
                    "link": link,
                    "scraped_at": datetime.now()
                })

    return pd.DataFrame(all_articles)


if __name__ == "__main__":
    df = scrape_conflict_news()

    print("\nTotal Relevant Articles:", len(df))
    df.describe()


Scraping BBC...

Scraping Associated Press...

Scraping UN News...

Scraping Al Jazeera...

Scraping Reuters...

Scraping Guardian...

Scraping CNN...

Scraping POLITICO...

Scraping NYTimes...

Scraping NDTV...

Scraping The Hindu...

Scraping Newswise...

Total Relevant Articles: 148


In [6]:
df['source'].value_counts()

source
Guardian            45
BBC                 39
CNN                 29
Al Jazeera          25
Associated Press    10
Name: count, dtype: int64

In [1]:
from ast import Global

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

RSS_SOURCES = {
    "BBC": "https://feeds.bbci.co.uk/news/rss.xml",}

KEYWORDS = [
    "war", "iran", "israel", "united states",
    "us", "missile", "attack", "military",
    "retaliation", "conflict", "gaza",
    "tehran", "hezbollah"
]

def isconflict(text):
    text=text.lower()
    keywords = ["conflict", "dispute", "issue", "problem"]
    return any (k in text for k in keywords)

def extractarticles(url):
    try:
        response=requests.get(url, header=HEADERS, timeout=10)
        soup=BeautifulSoup(response.text, 'html.parser')
        paragraph=soup.find_all('p')
        for p in paragraph:
            print(p.get_text())

    except Exception as e:
        print("Error scraping:", url)
        return ""

def scrape_conflict_news():
    all_articles=[]

    for source_name, rss_url in RSS_SOURCES.items():
        print(f"Scraping {source_name}..")
        feed = feedparser.parse(rss_url)
       

if __name__ == "__main__":
    scrape_conflict_news()

Scraping BBC..
